In [3]:
import numpy as np
import pandas as pd
import time

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout
)

### CARGAR DATASETS

In [4]:
datos = np.load("datos/gatos_perros_dataset.npz")

X_train = datos["X_train"]
y_train = datos["y_train"]

X_val = datos["X_val"]
y_val = datos["y_val"]

X_test = datos["X_test"]
y_test = datos["y_test"]


# Cargar datos de test por separado
datos_test = np.load("datos/datos-test.npz")

X_test_independiente = datos_test["X_test"]
y_test_independiente = datos_test["y_test"]


# Cargar datos de prueba individual
datos_individual = np.load("datos/datos-prueba-individual.npz")

X_individual = datos_individual["X_individual"]
y_individual = datos_individual["y_individual"]


print("Datos cargados correctamente:\n")

print(f"X_train:      {X_train.shape} | y_train:      {y_train.shape}")
print(f"X_val:        {X_val.shape} | y_val:        {y_val.shape}")
print(f"X_test:       {X_test.shape} | y_test:       {y_test.shape}")
print(
    f"X_test_ind.:  {X_test_independiente.shape} | "
    f"y_test_ind.:  {y_test_independiente.shape}"
)
print(
    f"X_individual: {X_individual.shape} | "
    f"y_individual: {y_individual.shape}"
)

Datos cargados correctamente:

X_train:      (13993, 784) | y_train:      (13993,)
X_val:        (2998, 784) | y_val:        (2998,)
X_test:       (2999, 784) | y_test:       (2999,)
X_test_ind.:  (2999, 784) | y_test_ind.:  (2999,)
X_individual: (10, 784) | y_individual: (10,)


### FUNCIONES AUXILIARES

In [5]:
def calcular_metricas(y_real, y_pred):
    """
    Calcula las métricas principales de clasificación.
    """

    return {
        "accuracy": accuracy_score(y_real, y_pred),
        "precision": precision_score(
            y_real,
            y_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_real,
            y_pred,
            zero_division=0
        ),
        "f1_score": f1_score(
            y_real,
            y_pred,
            zero_division=0
        )
    }

## CREAR CNN

In [6]:
X_train_cnn = X_train.reshape(-1, 28, 28, 1)
X_val_cnn = X_val.reshape(-1, 28, 28, 1)
X_test_cnn = X_test.reshape(-1, 28, 28, 1)

In [7]:
modelo_cnn = Sequential([
    Conv2D(
        32,
        (3, 3),
        activation="relu",
        input_shape=(28, 28, 1)
    ),

    MaxPooling2D((2, 2)),

    Conv2D(
        64,
        (3, 3),
        activation="relu"
    ),

    MaxPooling2D((2, 2)),

    Flatten(),

    Dense(
        128,
        activation="relu"
    ),

    Dropout(0.5),

    Dense(
        1,
        activation="sigmoid"
    )
])


modelo_cnn.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


modelo_cnn.summary()

c:\Users\Jpose75\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 223,873 (874.50 KB)

 Trainable params: 223,873 (874.50 KB)

 Non-trainable params: 0 (0.00 B)

### ENTRENAR CNN

In [8]:
inicio = time.perf_counter()

historial_cnn = modelo_cnn.fit(
    X_train_cnn,
    y_train,

    validation_data=(
        X_val_cnn,
        y_val
    ),

    epochs=15,
    batch_size=128,

    verbose=1
)

tiempo_entrenamiento_cnn = (
    time.perf_counter() - inicio
)

Epoch 1/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.7804 - loss: 0.4591 - val_accuracy: 0.8462 - val_loss: 0.3623
Epoch 2/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.8524 - loss: 0.3390 - val_accuracy: 0.8579 - val_loss: 0.3264
Epoch 3/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.8699 - loss: 0.3102 - val_accuracy: 0.8672 - val_loss: 0.3115
Epoch 4/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8772 - loss: 0.2890 - val_accuracy: 0.8662 - val_loss: 0.3096
Epoch 5/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8791 - loss: 0.2841 - val_accuracy: 0.8682 - val_loss: 0.2995
Epoch 6/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8862 - loss: 0.2692 - val_accuracy: 0.8742 - val_loss: 0.2974
Epoch 7/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8957 - loss: 0.2553 - val_accuracy: 0.8763 - val_loss: 0.2928
Epoch 8/15
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8993 - loss: 0.2420 - val_accu

### PREDICCION CNN

In [9]:
inicio = time.perf_counter()

probabilidades_cnn = modelo_cnn.predict(
    X_test_cnn,
    verbose=0
)

tiempo_prediccion_cnn = (
    time.perf_counter() - inicio
)


# Convertir probabilidades a clases
y_pred_cnn = (
    probabilidades_cnn.ravel() >= 0.5
).astype(int)

### MÉTRICAS CNN

In [10]:
metricas_cnn = calcular_metricas(
    y_test,
    y_pred_cnn
)

matriz_cnn = confusion_matrix(
    y_test,
    y_pred_cnn
)

## CREAR MLP

In [11]:
modelo_mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",

    max_iter=100,

    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,

    random_state=42
)

### ENTRENAR MLP

In [12]:
inicio = time.perf_counter()

modelo_mlp.fit(
    X_train,
    y_train
)

tiempo_entrenamiento_mlp = (
    time.perf_counter() - inicio
)

### 10. PREDICCION MLP

In [13]:
inicio = time.perf_counter()

y_pred_mlp = modelo_mlp.predict(
    X_test
)

tiempo_prediccion_mlp = (
    time.perf_counter() - inicio
)


### METRICAS MLP

In [14]:
metricas_mlp = calcular_metricas(
    y_test,
    y_pred_mlp
)

matriz_mlp = confusion_matrix(
    y_test,
    y_pred_mlp
)


## RESULTADOS GENERALES

In [15]:
resultados_generales_1 = pd.DataFrame([
    {
        "modelo": "CNN",

        "accuracy": metricas_cnn["accuracy"],
        "precision": metricas_cnn["precision"],
        "recall": metricas_cnn["recall"],
        "f1_score": metricas_cnn["f1_score"],

        "tiempo_entrenamiento_s":
            tiempo_entrenamiento_cnn,

        "tiempo_prediccion_s":
            tiempo_prediccion_cnn,

        "epocas":
            len(historial_cnn.history["loss"])
    },

    {
        "modelo": "MLP",

        "accuracy": metricas_mlp["accuracy"],
        "precision": metricas_mlp["precision"],
        "recall": metricas_mlp["recall"],
        "f1_score": metricas_mlp["f1_score"],

        "tiempo_entrenamiento_s":
            tiempo_entrenamiento_mlp,

        "tiempo_prediccion_s":
            tiempo_prediccion_mlp,

        "epocas":
            modelo_mlp.n_iter_
    }
])

In [16]:
print(
    resultados_generales_1.to_string(
        index=False
    )
)


print("\nMatriz de confusión CNN:")
print(matriz_cnn)

print("\nMatriz de confusión MLP:")
print(matriz_mlp)

modelo  accuracy  precision  recall  f1_score  tiempo_entrenamiento_s  tiempo_prediccion_s  epocas
   CNN  0.879627   0.888737   0.868  0.878246               21.766189             0.294414      15
   MLP  0.824942   0.789661   0.886  0.835061                2.148072             0.004855      16

Matriz de confusión CNN:
[[1336  163]
 [ 198 1302]]

Matriz de confusión MLP:
[[1145  354]
 [ 171 1329]]


## PREPARAR IMAGENES INDIVIDUALES

In [17]:
X_individual_normalizado = (
    X_individual.astype(np.float32) / 255.0
)

X_individual_mlp = X_individual_normalizado.reshape(
    len(X_individual_normalizado),
    -1
)

X_individual_cnn = X_individual_normalizado.reshape(
    -1,
    28,
    28,
    1
)

In [18]:
resultados_individuales = []


for i in range(len(X_individual)):

    # =======================
    # IMAGEN ORIGINAL

    imagen_original = X_individual[i].astype(
        np.uint8
    )

    imagen_original = imagen_original.reshape(
        28,
        28
    )


    # ======================================
    # CNN

    imagen_cnn = X_individual_cnn[i:i+1]

    inicio = time.perf_counter()

    prob_cnn = modelo_cnn.predict(
        imagen_cnn,
        verbose=0
    )[0][0]

    tiempo_cnn = (
        time.perf_counter() - inicio
    )

    pred_cnn = int(prob_cnn >= 0.5)


    # =====================
    # MLP

    imagen_mlp = X_individual_mlp[i:i+1]

    inicio = time.perf_counter()

    pred_mlp = int(
        modelo_mlp.predict(imagen_mlp)[0]
    )

    tiempo_mlp = (
        time.perf_counter() - inicio
    )


    # ================================

    resultados_individuales.append({

        "indice_imagen": i,

        "imagen": imagen_original,

        "etiqueta_real": int(
            y_individual[i]
        ),

        # CNN
        "prediccion_CNN": pred_cnn,

        "probabilidad_CNN": float(
            prob_cnn
        ),

        "tiempo_prediccion_CNN_s":
            tiempo_cnn,

        # MLP
        "prediccion_MLP": pred_mlp,

        "tiempo_prediccion_MLP_s":
            tiempo_mlp
    })

In [19]:
resultados_individuales_2 = pd.DataFrame(
    resultados_individuales
)

print(
    resultados_individuales_2[
        [
            "indice_imagen",
            "etiqueta_real",
            "prediccion_CNN",
            "probabilidad_CNN",
            "tiempo_prediccion_CNN_s",
            "prediccion_MLP",
            "tiempo_prediccion_MLP_s"
        ]
    ].to_string(index=False)
)

 indice_imagen  etiqueta_real  prediccion_CNN  probabilidad_CNN  tiempo_prediccion_CNN_s  prediccion_MLP  tiempo_prediccion_MLP_s
             0              0               0          0.000097                 0.065795               0                 0.000286
             1              1               1          0.998473                 0.052259               1                 0.000274
             2              0               0          0.000008                 0.056001               0                 0.000286
             3              1               1          0.957954                 0.051603               1                 0.000275
             4              0               0          0.000690                 0.049139               0                 0.000259
             5              0               0          0.000328                 0.049758               0                 0.000266
             6              1               1          0.722242                 0.055816  

## GUARDAR RESULTADOS

In [ ]:
resultados_generales_1.to_csv(
    "resultados/resultados_generales_1.csv",
    index=False
)

#==========================
# Guardamos las imágenes individuales

np.savez_compressed(
    "resultados/resultados_individuales_1.npz",

    imagenes=np.stack(
        resultados_individuales_2["imagen"].values
    ),

    etiquetas_reales=
        resultados_individuales_2[
            "etiqueta_real"
        ].values,

    predicciones_CNN=
        resultados_individuales_2[
            "prediccion_CNN"
        ].values,

    probabilidades_CNN=
        resultados_individuales_2[
            "probabilidad_CNN"
        ].values,

    tiempos_CNN=
        resultados_individuales_2[
            "tiempo_prediccion_CNN_s"
        ].values,

    predicciones_MLP=
        resultados_individuales_2[
            "prediccion_MLP"
        ].values,

    tiempos_MLP=
        resultados_individuales_2[
            "tiempo_prediccion_MLP_s"
        ].values
)
